# Riyadh Air Quality — Real Historical Data
Downloads model-derived historical PM2.5/PM10 and historical weather for Riyadh, Saudi Arabia, for 2024. No synthetic data. Open-Meteo/CAMS air quality is gridded model data, not ground-station measurements. Verify API terms and attribution before publication.

In [ ]:
!pip -q install requests pandas openpyxl

In [ ]:
import requests
import pandas as pd
from google.colab import files

LAT, LON = 24.7136, 46.6753
START, END = "2024-01-01", "2024-12-31"
TZ = "Asia/Riyadh"

def fetch(url, params):
    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()
    data = response.json()
    if data.get("error"):
        raise RuntimeError(data.get("reason", "API error"))
    return data


In [ ]:
air = fetch("https://air-quality-api.open-meteo.com/v1/air-quality", {
    "latitude": LAT, "longitude": LON,
    "start_date": START, "end_date": END,
    "hourly": "pm2_5,pm10", "timezone": TZ,
})
air_df = pd.DataFrame(air["hourly"]).rename(columns={"time":"datetime"})
print("Air quality rows:",len(air_df)); display(air_df.head())

In [ ]:
weather = fetch("https://archive-api.open-meteo.com/v1/archive", {
    "latitude": LAT, "longitude": LON,
    "start_date": START, "end_date": END,
    "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m",
    "timezone": TZ,
})
weather_df = pd.DataFrame(weather["hourly"]).rename(columns={"time":"datetime"})
print("Weather rows:",len(weather_df)); display(weather_df.head())

In [ ]:
df = air_df.merge(weather_df, on="datetime", how="inner", validate="one_to_one")
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)
print("Merged rows:", len(df))
print("Missing values:\n", df.isna().sum())
display(df.head())

## Predict tomorrow’s PM2.5 (24 hours ahead)
The target is shifted 24 hours forward. Features only use current/past information. Use chronological train/test split later, not random split.

In [ ]:
df["pm2_5_next_24h"] = df["pm2_5"].shift(-24)
df["hour"] = df["datetime"].dt.hour
df["month"] = df["datetime"].dt.month
model_df = df.dropna(subset=["pm2_5_next_24h"]).copy()
print("Rows ready for modelling:", len(model_df))
display(model_df.head())

In [ ]:
sources = pd.DataFrame([
    {"dataset":"Air quality", "provider":"Open-Meteo / CAMS", "url":"https://open-meteo.com/en/docs/air-quality-api", "type":"Model-derived, not ground monitor", "period":"2024", "location":"Riyadh, Saudi Arabia"},
    {"dataset":"Weather", "provider":"Open-Meteo historical weather", "url":"https://open-meteo.com/en/docs/historical-weather-api", "type":"Reanalysis/model-derived", "period":"2024", "location":"Riyadh, Saudi Arabia"},
])
output="Riyadh_Air_Quality_2024_Real_Data.xlsx"
with pd.ExcelWriter(output, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Hourly_Data", index=False)
    model_df.to_excel(writer, sheet_name="ML_Ready", index=False)
    sources.to_excel(writer, sheet_name="Sources", index=False)
print("Created:", output)
files.download(output)